<div dir="rtl" style="text-align: right;">

#  آشنایی با API وی‌ویت (Weaviate) — نسخه‌ی Google Colab با Weaviate Cloud و سرویس ابری Jina

<div align="center">
  <img src="./images/weaviate.png" alt="Weaviate" width="10%">
</div>


1. به‌جای Weaviate لوکال با Docker، به یک کلاستر **Weaviate Cloud (رایگان)** وصل می‌شویم.
2. به‌جای مدل embedding لوکال `bge-m3` (که از طریق Ollama سرو می‌شد)، از **سرویس ابری Jina AI** یک مدل embedding معمولی (تک‌برداری) به اسم `jina-embeddings-v3` هم دارد که **چندزبانه** است (۸۹ زبان از جمله فارسی) — همان چیزی که این‌جا لازم داریم.


خروجی این نوت‌بوک همان مباحث نوت‌بوک اصلی است: ساخت Collection، پیکربندی Vectorizer، افزودن داده، فیلترها، جستجوی معنایی، BM25 و جستجوی ترکیبی — با تمرکز بر تست فارسی.

بزن بریم!

</div>

<div dir="rtl" style="text-align: right;">

<a id='1'></a>
## ۱ - پیش‌نیازها
---

دو چیز لازم است:

- یک کلاستر **رایگان** روی [console.weaviate.cloud](https://console.weaviate.cloud) با **REST Endpoint** و **Admin API Key**.
- یک کلید API رایگان از [jina.ai](https://jina.ai).

هر سه مقدار زیر را در بخش **Secrets** کولب (آیکن 🔑 در نوار کناری چپ) ذخیره کنید و toggle «Notebook access» را برایشان روشن کنید:

- `WEAVIATE_URL`
- `WEAVIATE_API_KEY`
- `JINA_API_KEY`

</div>

In [1]:
!pip install -q -U weaviate-client tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 652.7/652.7 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 100.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 kB 3.2 MB/s eta 0:00:00


<div dir="rtl" style="text-align: right;">

<a id='2'></a>
## ۲ - اتصال به Weaviate Cloud
---

کلید Jina را به‌عنوان هدر `X-JinaAI-Api-Key` می‌دهیم تا هر وقت Weaviate به یک embedding نیاز داشت (چه هنگام افزودن داده، چه هنگام جستجو)، خودش از Jina بگیرد.

</div>

In [3]:
from google.colab import userdata
import weaviate
from weaviate.classes.init import Auth
from weaviate.classes.config import Configure, Property, DataType
from weaviate.classes.query import Filter, MetadataQuery
from weaviate.util import generate_uuid5
from tqdm import tqdm

WEAVIATE_URL     = userdata.get('WEAVIATE_URL')
WEAVIATE_API_KEY = userdata.get('WEAVIATE_API_KEY')
JINA_API_KEY     = userdata.get('JINA_API_KEY')

client = weaviate.connect_to_weaviate_cloud(
    cluster_url=WEAVIATE_URL,
    auth_credentials=Auth.api_key(WEAVIATE_API_KEY),
    headers={"X-JinaAI-Api-Key": JINA_API_KEY},
)

print("اتصال برقرار است:", client.is_ready())

اتصال برقرار است: True


<div dir="rtl" style="text-align: right;">

<a id='3'></a>
## ۳ - دیتاست فارسی نمونه
---

همان الگوی دیتاست گردشگری نوت‌بوک اصلی، با فیلدهای بیشتر (فصل مناسب سفر، جاذبه‌ها، امتیاز کاربران و تاریخ به‌روزرسانی) تا فیلترها و جستجوی ترکیبی را هم بتوانیم نشان دهیم.

</div>

In [4]:
data = [
    {
        "place": "دره ستارگان",
        "state": "هرمزگان",
        "description": "دره‌ای شگفت‌انگیز در جزیره قشم با صخره‌های نمکی و فرسایشی که منظره‌ای شبیه سطح ماه دارد.",
        "best_season_to_visit": "پاییز, زمستان",
        "attractions": "صخره‌های ستاره‌ای, تورهای شب‌نوردی",
        "budget": "ارزان",
        "user_ratings": 4.6,
    },
    {
        "place": "کویر مرنجاب",
        "state": "اصفهان",
        "description": "کویری خاموش و دیدنی با تپه‌های شنی و دریاچه نمک، مناسب برای سفرهای صحرایی و اقامت شبانه در کمپ.",
        "best_season_to_visit": "پاییز, زمستان",
        "attractions": "کاروانسرای مرنجاب, دریاچه نمک",
        "budget": "متوسط",
        "user_ratings": 4.4,
    },
    {
        "place": "نقش رستم",
        "state": "فارس",
        "description": "مجموعه‌ای از آرامگاه‌های صخره‌ای و نقش‌برجسته‌های شاهان هخامنشی و ساسانی.",
        "best_season_to_visit": "بهار, پاییز",
        "attractions": "آرامگاه داریوش, نقش‌برجسته‌های ساسانی",
        "budget": "ارزان",
        "user_ratings": 4.7,
    },
    {
        "place": "جزیره هرمز",
        "state": "هرمزگان",
        "description": "جزیره‌ای رنگارنگ با خاک‌های اُخرایی، معروف به ساحل رنگین‌کمان.",
        "best_season_to_visit": "زمستان",
        "attractions": "ساحل رنگین‌کمان, غار نمکی",
        "budget": "متوسط",
        "user_ratings": 4.6,
    },
    {
        "place": "چشمه‌های آب‌گرم سرعین",
        "state": "اردبیل",
        "description": "منطقه‌ای معروف به چشمه‌های آب‌گرم طبیعی و درمانی در دامنه کوه سبلان.",
        "best_season_to_visit": "تابستان, پاییز",
        "attractions": "چشمه آب‌گرم گاومیش‌گلی, حمام‌های سنتی",
        "budget": "متوسط",
        "user_ratings": 4.4,
    },
    {
        "place": "پل خواجو",
        "state": "اصفهان",
        "description": "پلی تاریخی و چندمنظوره روی زاینده‌رود با معماری دوران صفوی.",
        "best_season_to_visit": "بهار, پاییز",
        "attractions": "طاق‌های تزئینی, نشستن کنار رودخانه",
        "budget": "ارزان",
        "user_ratings": 4.8,
    },
    {
        "place": "جزیره کیش",
        "state": "هرمزگان",
        "description": "جزیره‌ای مرجانی در خلیج فارس، معروف به سواحل زیبا و مراکز خرید.",
        "best_season_to_visit": "زمستان",
        "attractions": "کشتی یونانی, دلفیناریوم",
        "budget": "گران",
        "user_ratings": 4.5,
    },
    {
        "place": "تنگه واشی",
        "state": "لرستان",
        "description": "دره‌ای صخره‌ای و سرسبز با آبشارهای فصلی و مسیرهای کوهنوردی.",
        "best_season_to_visit": "بهار",
        "attractions": "آبشارهای فصلی, پل‌های معلق",
        "budget": "ارزان",
        "user_ratings": 4.7,
    },
]

for d in data:
    print(d["place"], "-", d["description"][:40], "...")

دره ستارگان - دره‌ای شگفت‌انگیز در جزیره قشم با صخره‌ه ...
کویر مرنجاب - کویری خاموش و دیدنی با تپه‌های شنی و دری ...
نقش رستم - مجموعه‌ای از آرامگاه‌های صخره‌ای و نقش‌ب ...
جزیره هرمز - جزیره‌ای رنگارنگ با خاک‌های اُخرایی، معر ...
چشمه‌های آب‌گرم سرعین - منطقه‌ای معروف به چشمه‌های آب‌گرم طبیعی  ...
پل خواجو - پلی تاریخی و چندمنظوره روی زاینده‌رود با ...
جزیره کیش - جزیره‌ای مرجانی در خلیج فارس، معروف به س ...
تنگه واشی - دره‌ای صخره‌ای و سرسبز با آبشارهای فصلی  ...


<div dir="rtl" style="text-align: right;">

<a id='4'></a>
## ۴ - پیکربندی Vectorizer (سرویس ابری Jina)
---

مدل `jina-embeddings-v3` یک مدل embedding **معمولی/تک‌برداری** (نه ColBERT/multi-vector) و **چندزبانه** است (۸۹ زبان از جمله فارسی). چون تک‌برداری است، هیچ محدودیتی مثل ماجرای `hfresh`/ColBERT ندارد و روی ایندکس پیش‌فرض `hnsw` هم بدون مشکل کار می‌کند.

</div>

In [5]:
vectorizer_config = [
    Configure.Vectors.text2vec_jinaai(
        name="vector",
        source_properties=[
            "place", "state", "description",
            "best_season_to_visit", "attractions", "budget",
        ],
        model="jina-embeddings-v3",
        vectorize_collection_name=False,
    )
]

<div dir="rtl" style="text-align: right;">

<a id='5'></a>
## ۵ - ساخت Collection و افزودن داده
---

</div>

In [6]:
COLLECTION_NAME = "example_collection_jina"

if client.collections.exists(COLLECTION_NAME):
    client.collections.delete(COLLECTION_NAME)

collection = client.collections.create(
    name=COLLECTION_NAME,
    vector_config=vectorizer_config,
    properties=[
        Property(name="place", vectorize_property_name=True, data_type=DataType.TEXT),
        Property(name="state", vectorize_property_name=True, data_type=DataType.TEXT),
        Property(name="description", vectorize_property_name=True, data_type=DataType.TEXT),
        Property(name="best_season_to_visit", vectorize_property_name=True, data_type=DataType.TEXT),
        Property(name="attractions", vectorize_property_name=True, data_type=DataType.TEXT),
        Property(name="budget", vectorize_property_name=True, data_type=DataType.TEXT),
        Property(name="user_ratings", data_type=DataType.NUMBER),
    ],
)

with collection.batch.fixed_size(batch_size=4, concurrent_requests=2) as batch:
    for d in tqdm(data):
        batch.add_object(properties=d, uuid=generate_uuid5(d))

if collection.batch.failed_objects:
    print("⚠️ برخی اسناد افزوده نشدند:")
    print(collection.batch.failed_objects[0])

print("تعداد اسناد ذخیره‌شده:", len(collection))

100%|██████████| 8/8 [00:00<00:00, 741.45it/s]


تعداد اسناد ذخیره‌شده: 8


<div dir="rtl" style="text-align: right;">

<a id='6'></a>
## ۶ - جستجوی معنایی (Semantic / near_text) — تست فارسی
---

چون از model integration استفاده کردیم، کافی است متن فارسی کوئری را مستقیم به `near_text` بدهیم؛ Weaviate خودش آن را با Jina امبد می‌کند.

</div>

In [7]:
response = collection.query.near_text(
    query="جایی آرام برای فرار از شلوغی شهر و طبیعت‌گردی",
    limit=3,
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print(f"فاصله: {obj.metadata.distance:.4f}")
    for k, v in obj.properties.items():
        print(f"  {k}: {v}")
    print("---")

فاصله: 0.5996
  description: کویری خاموش و دیدنی با تپه‌های شنی و دریاچه نمک، مناسب برای سفرهای صحرایی و اقامت شبانه در کمپ.
  best_season_to_visit: پاییز, زمستان
  attractions: کاروانسرای مرنجاب, دریاچه نمک
  budget: متوسط
  state: اصفهان
  place: کویر مرنجاب
  user_ratings: 4.4
---
فاصله: 0.6915
  description: دره‌ای صخره‌ای و سرسبز با آبشارهای فصلی و مسیرهای کوهنوردی.
  best_season_to_visit: بهار
  attractions: آبشارهای فصلی, پل‌های معلق
  budget: ارزان
  state: لرستان
  place: تنگه واشی
  user_ratings: 4.7
---
فاصله: 0.7458
  description: منطقه‌ای معروف به چشمه‌های آب‌گرم طبیعی و درمانی در دامنه کوه سبلان.
  best_season_to_visit: تابستان, پاییز
  attractions: چشمه آب‌گرم گاومیش‌گلی, حمام‌های سنتی
  budget: متوسط
  state: اردبیل
  place: چشمه‌های آب‌گرم سرعین
  user_ratings: 4.4
---


<div dir="rtl" style="text-align: right;">

<a id='7'></a>
## ۷ - فیلترها
---

می‌توانیم جستجوی معنایی را با فیلتر روی ویژگی‌های ساختاریافته ترکیب کنیم؛ مثلاً فقط مقاصد ارزان.

</div>

In [8]:
response = collection.query.near_text(
    query="جایی تاریخی و فرهنگی برای بازدید",
    filters=Filter.by_property("budget").equal("ارزان"),
    limit=3,
)

for obj in response.objects:
    for k, v in obj.properties.items():
        print(f"  {k}: {v}")
    print("---")

  description: پلی تاریخی و چندمنظوره روی زاینده‌رود با معماری دوران صفوی.
  best_season_to_visit: بهار, پاییز
  attractions: طاق‌های تزئینی, نشستن کنار رودخانه
  budget: ارزان
  state: اصفهان
  place: پل خواجو
  user_ratings: 4.8
---
  description: دره‌ای شگفت‌انگیز در جزیره قشم با صخره‌های نمکی و فرسایشی که منظره‌ای شبیه سطح ماه دارد.
  best_season_to_visit: پاییز, زمستان
  budget: ارزان
  attractions: صخره‌های ستاره‌ای, تورهای شب‌نوردی
  state: هرمزگان
  place: دره ستارگان
  user_ratings: 4.6
---
  description: مجموعه‌ای از آرامگاه‌های صخره‌ای و نقش‌برجسته‌های شاهان هخامنشی و ساسانی.
  best_season_to_visit: بهار, پاییز
  budget: ارزان
  attractions: آرامگاه داریوش, نقش‌برجسته‌های ساسانی
  state: فارس
  place: نقش رستم
  user_ratings: 4.7
---


<div dir="rtl" style="text-align: right;">

<a id='8'></a>
## ۸ - جستجوی کلیدواژه‌ای (BM25) — تست فارسی
---

BM25 برخلاف جستجوی معنایی، به هیچ embedding نیاز ندارد و صرفاً روی تطبیق کلمات کار می‌کند؛ برای همین حتی بدون کلید Jina هم کار می‌کند.

</div>

In [9]:
response = collection.query.bm25(
    query="کویر شنی",
    limit=3,
)

for obj in response.objects:
    for k, v in obj.properties.items():
        print(f"  {k}: {v}")
    print("---")

  description: کویری خاموش و دیدنی با تپه‌های شنی و دریاچه نمک، مناسب برای سفرهای صحرایی و اقامت شبانه در کمپ.
  best_season_to_visit: پاییز, زمستان
  attractions: کاروانسرای مرنجاب, دریاچه نمک
  budget: متوسط
  state: اصفهان
  place: کویر مرنجاب
  user_ratings: 4.4
---


<div dir="rtl" style="text-align: right;">

<a id='9'></a>
## ۹ - جستجوی ترکیبی (Hybrid) — تست فارسی
---

جستجوی ترکیبی، نتایج BM25 و جستجوی معنایی را با روش RRF ادغام می‌کند. پارامتر `alpha` مشخص می‌کند چه وزنی به هرکدام داده شود (۰ = فقط BM25، ۱ = فقط معنایی).

</div>

In [10]:
response = collection.query.hybrid(
    query="می‌خواهم برای سفر زمستانی پیشنهاد بگیرم؛ به دنبال مکان‌های ارزان هستم",
    filters=Filter.by_property("budget").contains_any(["ارزان", "متوسط"]),
    alpha=0.3,
    limit=4,
)

for obj in response.objects:
    for k, v in obj.properties.items():
        print(f"  {k}: {v}")
    print("---")

  description: منطقه‌ای معروف به چشمه‌های آب‌گرم طبیعی و درمانی در دامنه کوه سبلان.
  best_season_to_visit: تابستان, پاییز
  attractions: چشمه آب‌گرم گاومیش‌گلی, حمام‌های سنتی
  budget: متوسط
  state: اردبیل
  place: چشمه‌های آب‌گرم سرعین
  user_ratings: 4.4
---
  description: کویری خاموش و دیدنی با تپه‌های شنی و دریاچه نمک، مناسب برای سفرهای صحرایی و اقامت شبانه در کمپ.
  best_season_to_visit: پاییز, زمستان
  attractions: کاروانسرای مرنجاب, دریاچه نمک
  budget: متوسط
  state: اصفهان
  place: کویر مرنجاب
  user_ratings: 4.4
---
  description: مجموعه‌ای از آرامگاه‌های صخره‌ای و نقش‌برجسته‌های شاهان هخامنشی و ساسانی.
  best_season_to_visit: بهار, پاییز
  attractions: آرامگاه داریوش, نقش‌برجسته‌های ساسانی
  budget: ارزان
  state: فارس
  place: نقش رستم
  user_ratings: 4.7
---
  description: دره‌ای شگفت‌انگیز در جزیره قشم با صخره‌های نمکی و فرسایشی که منظره‌ای شبیه سطح ماه دارد.
  best_season_to_visit: پاییز, زمستان
  attractions: صخره‌های ستاره‌ای, تورهای شب‌نوردی
  budget: ارزان
  state:

<div dir="rtl" style="text-align: right;">

<a id='10'></a>
## ۱۰ - جمع‌بندی
---

در این نسخه یاد گرفتید:
- چطور بدون Docker، مستقیماً به یک کلاستر رایگان روی Weaviate Cloud وصل شوید
- چطور به‌جای مدل لوکال (Ollama/bge-m3)، از سرویس ابری Jina (`jina-embeddings-v3`) به‌عنوان vectorizer معمولی (تک‌برداری) استفاده کنید — با پشتیبانی کامل فارسی


</div>

In [11]:
# فراموش نکنید کلاینت را ببندید!
client.close()